<a href="https://colab.research.google.com/github/AKBER-HUSSAIN/DL-2026/blob/main/dlweek_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Week 12**

**RNN for next character prediction**

In [ ]:
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense
import numpy as np

# vocabulary
vocab = ['d','e','p','<stop>']

char2idx = {c:i for i,c in enumerate(vocab)}
idx2char = {i:c for c,i in char2idx.items()}

# sequence
sequence = ['d','e','p']
target = ['e','p','<stop>']

# one hot encoding
X = np.eye(len(vocab))[[char2idx[c] for c in sequence]]
X = X.reshape(len(sequence),1,len(vocab))

y = np.eye(len(vocab))[[char2idx[c] for c in target]]

# model
model = Sequential([

    SimpleRNN(8,input_shape=(1,len(vocab))),
    Dense(len(vocab),activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam'
)

# train
model.fit(X,y,epochs=200,verbose=0)

# predict next char after 'p'
test = np.eye(len(vocab))[char2idx['p']]
test = test.reshape(1,1,len(vocab))

pred = model.predict(test)

print("Next character:",idx2char[np.argmax(pred)])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 407ms/step
Next character: <stop>


**RNN next word prediction**

In [ ]:
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense
import numpy as np

# vocabulary
vocab = ['deep','learning','is','fun','<stop>']

word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

# sequence
sequence = ['deep','learning','is','fun']
target = ['learning','is','fun','<stop>']

# encoding
X = np.eye(len(vocab))[[word2idx[w] for w in sequence]]
X = X.reshape(len(sequence),1,len(vocab))

y = np.eye(len(vocab))[[word2idx[w] for w in target]]

# model
model = Sequential([

    SimpleRNN(10,input_shape=(1,len(vocab))),
    Dense(len(vocab),activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam'
)

# train
model.fit(X,y,epochs=300,verbose=0)

# test
test_word = 'is'

test = np.eye(len(vocab))[word2idx[test_word]]
test = test.reshape(1,1,len(vocab))

pred = model.predict(test)

print("Next word:",idx2word[np.argmax(pred)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 407ms/step
Next word: fun


**RNN sentence classification**

In [ ]:
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense, Embedding
import numpy as np

# vocabulary
vocab = {

    'this':0,
    'movie':1,
    'is':2,
    'great':3,
    'bad':4,
    'terrible':5
}

vocab_size = len(vocab)

# dataset
X = np.array([

    [0,1,2,3],   # positive
    [0,1,2,5]    # negative

])

y = np.array([1,0])

# model
model = Sequential([

    Embedding(vocab_size,8,input_length=4),

    SimpleRNN(16),

    Dense(1,activation='sigmoid')
])

model.compile(

    optimizer='adam',

    loss='binary_crossentropy',

    metrics=['accuracy']
)

# train
model.fit(X,y,epochs=100,verbose=0)

# test sentence
test = np.array([[0,1,2,4]])   # this movie is bad

pred = model.predict(test)

print(

    "Sentiment:",

    "Positive" if pred[0][0]>0.5 else "Negative"
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Sentiment: Negative


**LSTM for next word prediction**

In [ ]:
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense
import numpy as np

# vocabulary
vocab = {
    'the':0,
    'sun':1,
    'is':2,
    'shining':3,
    'bright':4,
    '<stop>':5
}

idx2word = {i:w for w,i in vocab.items()}

# training data
X = np.array([[0,1,2,3]])   # the sun is shining
y = np.array([4])           # bright

# model
model = Sequential([

    Embedding(len(vocab),10,input_length=4),

    LSTM(32),

    Dense(len(vocab),activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy'
)

# train
model.fit(X,y,epochs=200,verbose=0)

# test
test = np.array([[0,1,2,3]])

pred = model.predict(test)

print(
    "Next word:",
    idx2word[np.argmax(pred)]
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step
Next word: bright


**GRU for next word prediction**

In [ ]:
from keras.models import Sequential
from keras.layers import Embedding, GRU, Dense
import numpy as np

# vocabulary
vocab = {
    'machine':0,
    'learning':1,
    'is':2,
    'powerful':3,
    'tool':4,
    '<stop>':5
}

idx2word = {i:w for w,i in vocab.items()}

# training data
X = np.array([[0,1,2,3]])   # machine learning is powerful
y = np.array([4])           # tool

# model
model = Sequential([

    Embedding(len(vocab),10,input_length=4),

    GRU(32),

    Dense(len(vocab),activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy'
)

# train
model.fit(X,y,epochs=200,verbose=0)

# test
test = np.array([[0,1,2,3]])

pred = model.predict(test)

print(
    "Next word:",
    idx2word[np.argmax(pred)]
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
Next word: tool


**Encoder - Decoder model for translation**

In [ ]:
from keras.models import Model
from keras.layers import Input, SimpleRNN, Dense
import numpy as np

# simple pair
input_text = "how are you"
target_text = "మీరు ఎలా ఉన్నారు"

# char sets
input_chars = sorted(set(input_text))
target_chars = sorted(set(target_text))

in_idx = {c:i for i,c in enumerate(input_chars)}
tar_idx = {c:i for i,c in enumerate(target_chars)}

rev_tar_idx = {i:c for c,i in tar_idx.items()}

# one hot function
def one_hot(text, mapping):

    data = np.zeros((1,len(text),len(mapping)))

    for i,ch in enumerate(text):

        data[0,i,mapping[ch]] = 1

    return data

# encode
enc_input = one_hot(input_text,in_idx)

dec_input = one_hot(target_text,tar_idx)

dec_target = np.copy(dec_input)

# encoder
enc_in = Input(shape=(None,len(in_idx)))

_,state = SimpleRNN(16,return_state=True)(enc_in)

# decoder
dec_in = Input(shape=(None,len(tar_idx)))

dec_out,_ = SimpleRNN(

    16,

    return_sequences=True,

    return_state=True

)(dec_in,initial_state=state)

dec_out = Dense(len(tar_idx),activation='softmax')(dec_out)

# model
model = Model([enc_in,dec_in],dec_out)

model.compile(

    optimizer='adam',

    loss='categorical_crossentropy'
)

# train
model.fit(

    [enc_input,dec_input],

    dec_target,

    epochs=200,

    verbose=0
)

# predict
output = model.predict([enc_input,dec_input])

translated = ''.join(

    rev_tar_idx[np.argmax(i)]

    for i in output[0]
)

print("Input:",input_text)

print("Predicted:",translated)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 529ms/step
Input: how are you 
Predicted: మీరు ఎలా ఉన్నారు


In [ ]:
import tensorflow as tf
import numpy as np
from keras.layers import Layer

# Simple Attention Layer
class Attention(Layer):

    def call(self, inputs):

        query, values = inputs

        # similarity score
        score = tf.matmul(query, values, transpose_b=True)

        # attention weights
        weights = tf.nn.softmax(score)

        # context vector
        context = tf.matmul(weights, values)

        return context, weights
# dummy sequence data
# batch_size=1, time_steps=3, features=4

query = tf.constant([[
    [1.,0.,1.,0.],
    [0.,1.,0.,1.],
    [1.,1.,0.,0.]
]])

values = tf.constant([[
    [1.,2.,3.,4.],
    [2.,3.,4.,5.],
    [3.,4.,5.,6.]
]])

# apply attention
attention = Attention()

context, weights = attention([query, values])

print("Context Vector:\n", context.numpy())

print("\nAttention Weights:\n", weights.numpy())

Context Vector:
 [[[2.8509371 3.8509371 4.8509374 5.850937 ]
  [2.8509371 3.8509371 4.8509374 5.850937 ]
  [2.8509371 3.8509371 4.8509374 5.850937 ]]]

Attention Weights:
 [[[0.01587624 0.11731043 0.86681336]
  [0.01587624 0.11731043 0.86681336]
  [0.01587624 0.11731043 0.86681336]]]


In [ ]:
# The code creates an Attention layer that helps the model focus on the most important parts of the input data
# It compares the query with all values to calculate how relevant each part of the input is
# Based on this relevance, the model assigns weights (importance scores) using the softmax function
# Finally, it generates a context vector, which is a weighted combination of important information from the input sequence